# CatBoost-модель: самый прибыльный район по часу

Задача:

```text
для каждого часа предсказать район посадки, где будет максимальная выручка
```

Логика:

1. Читаем `my_clean_3_with_weather.parquet` батчами.
2. Строим таблицу:

```text
date_hour × PULocationID
```

3. Для каждой пары считаем:

```text
demand  = количество поездок
revenue = сумма total_amount
```

4. Если в районе в этот час поездок не было:

```text
demand = 0
revenue = 0
```

5. Обучаем градиентный бустинг `CatBoostRegressor` предсказывать `revenue`.
6. Для каждого часа выбираем район с максимальной `predicted_revenue`.

Почему CatBoost:

- это градиентный бустинг;
- хорошо работает с табличными данными;
- умеет работать с категориальными признаками;
- `PULocationID` можно не превращать в число “по смыслу”, а передать как категорию.

## 1. Установка CatBoost

In [ ]:
# Если catboost не установлен, раскомментируй и запусти:
!pip install catboost

## 2. Импорт библиотек

In [ ]:
import gc
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from catboost import CatBoostRegressor, Pool

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 3. Пути проекта

In [ ]:
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "my_clean_3_with_weather.parquet"

MODELS_DIR = PROJECT_ROOT / "models"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
print("Файл существует:", DATA_PATH.exists())
print("MODELS_DIR:", MODELS_DIR)
print("TABLES_DIR:", TABLES_DIR)

## 4. Настройки

In [ ]:
BATCH_SIZE = 150_000

# None = читать весь parquet.
# Для быстрой проверки можно поставить 3 или 5.
MAX_BATCHES = None

START_DATE = pd.Timestamp("2023-01-01 00:00:00")
END_DATE = pd.Timestamp("2025-01-01 00:00:00")

# Если обучение на всём train идёт слишком долго, можно поставить, например, 0.5 или 0.3.
# Для финального варианта лучше оставить 1.0.
TRAIN_SAMPLE_FRAC = 1.0

RANDOM_STATE = 42

## 5. Читаем parquet батчами и агрегируем

Нам нужны:

```text
tpep_pickup_datetime
PULocationID
total_amount
temperature
precipitation
snowfall
weather_code
PU_Zone
```

`PU_Zone` нужен только для красивого вывода результата.

In [ ]:
needed_columns = [
    "tpep_pickup_datetime",
    "PULocationID",
    "total_amount",
    "temperature",
    "precipitation",
    "snowfall",
    "weather_code",
    "PU_Zone",
]

weather_features = [
    "temperature",
    "precipitation",
    "snowfall",
    "weather_code",
]

parquet_file = pq.ParquetFile(DATA_PATH)

available_columns = parquet_file.schema.names

needed_columns = [col for col in needed_columns if col in available_columns]
weather_features = [col for col in weather_features if col in available_columns]

required_columns = ["tpep_pickup_datetime", "PULocationID", "total_amount"]
missing_columns = [col for col in required_columns if col not in needed_columns]

if missing_columns:
    raise ValueError(f"В parquet не хватает нужных колонок: {missing_columns}")

print("Будем читать колонки:")
print(needed_columns)

print("Погодные признаки:")
print(weather_features)

## 6. Батчевое чтение и первичная агрегация

Каждый батч сразу превращается в таблицу:

```text
date_hour | PULocationID | demand | revenue | weather_sum
```

Так мы не держим весь исходный датасет в памяти.

In [ ]:
batch_aggregates = []
pu_zone_lookups = []

print("Начинаем читать parquet батчами...")

for batch_number, batch in enumerate(
    parquet_file.iter_batches(
        batch_size=BATCH_SIZE,
        columns=needed_columns
    ),
    start=1
):
    if MAX_BATCHES is not None and batch_number > MAX_BATCHES:
        break

    part = batch.to_pandas()

    part["tpep_pickup_datetime"] = pd.to_datetime(
        part["tpep_pickup_datetime"],
        errors="coerce"
    )

    part["date_hour"] = part["tpep_pickup_datetime"].dt.floor("h")

    part["PULocationID"] = pd.to_numeric(
        part["PULocationID"],
        errors="coerce"
    )

    part["total_amount"] = pd.to_numeric(
        part["total_amount"],
        errors="coerce"
    )

    for col in weather_features:
        part[col] = pd.to_numeric(part[col], errors="coerce")

    mask = (
        part["date_hour"].notna()
        & part["PULocationID"].notna()
        & part["total_amount"].notna()
        & (part["date_hour"] >= START_DATE)
        & (part["date_hour"] < END_DATE)
        & (part["total_amount"] >= 0)
    )

    part = part.loc[mask]

    if len(part) == 0:
        del part, mask
        gc.collect()
        continue

    part["PULocationID"] = part["PULocationID"].astype("int16")
    part["total_amount"] = part["total_amount"].astype("float32")

    for col in weather_features:
        part[col] = part[col].astype("float32")

    if "PU_Zone" in part.columns:
        pu_zone_lookups.append(
            part[["PULocationID", "PU_Zone"]]
            .dropna()
            .drop_duplicates("PULocationID")
        )

    part_small = part[
        ["date_hour", "PULocationID", "total_amount"] + weather_features
    ].copy()

    for col in weather_features:
        part_small[col + "_sum"] = part_small[col]

    agg_dict = {
        "demand": ("PULocationID", "size"),
        "revenue": ("total_amount", "sum"),
    }

    for col in weather_features:
        agg_dict[col + "_sum"] = (col + "_sum", "sum")

    grouped = (
        part_small
        .groupby(["date_hour", "PULocationID"])
        .agg(**agg_dict)
        .reset_index()
    )

    grouped["PULocationID"] = grouped["PULocationID"].astype("int16")
    grouped["demand"] = grouped["demand"].astype("int32")
    grouped["revenue"] = grouped["revenue"].astype("float32")

    for col in grouped.columns:
        if col.endswith("_sum"):
            grouped[col] = grouped[col].astype("float32")

    batch_aggregates.append(grouped)

    print(
        f"Батч {batch_number}: "
        f"строк после фильтра = {len(part):,}, "
        f"агрегированных строк = {len(grouped):,}"
    )

    del part, part_small, grouped, mask
    gc.collect()

print("Батчи прочитаны.")
print("Количество агрегированных батчей:", len(batch_aggregates))

## 7. Финальная агрегация батчей

In [ ]:
if len(batch_aggregates) == 0:
    raise ValueError("После фильтрации данных не осталось.")

hourly_sum = pd.concat(batch_aggregates, ignore_index=True)

sum_columns = [col for col in hourly_sum.columns if col.endswith("_sum")]

aggregation_rules = {
    "demand": "sum",
    "revenue": "sum",
}

for col in sum_columns:
    aggregation_rules[col] = "sum"

hourly_real = (
    hourly_sum
    .groupby(["date_hour", "PULocationID"], as_index=False)
    .agg(aggregation_rules)
)

# Среднюю погоду восстанавливаем как сумма / demand.
for col in weather_features:
    sum_col = col + "_sum"
    if sum_col in hourly_real.columns:
        hourly_real[col] = (
            hourly_real[sum_col] / hourly_real["demand"]
        ).astype("float32")

hourly_real = hourly_real.drop(columns=sum_columns)

hourly_real["PULocationID"] = hourly_real["PULocationID"].astype("int16")
hourly_real["demand"] = hourly_real["demand"].astype("int32")
hourly_real["revenue"] = hourly_real["revenue"].astype("float32")

print("Размер hourly_real:", hourly_real.shape)
print("Минимальная дата:", hourly_real["date_hour"].min())
print("Максимальная дата:", hourly_real["date_hour"].max())

display(hourly_real.head())

del hourly_sum
del batch_aggregates
gc.collect()

## 8. Справочник районов

In [ ]:
if pu_zone_lookups:
    pu_zone_lookup = (
        pd.concat(pu_zone_lookups, ignore_index=True)
        .drop_duplicates("PULocationID")
    )
    pu_zone_lookup["PULocationID"] = pu_zone_lookup["PULocationID"].astype("int16")
else:
    pu_zone_lookup = pd.DataFrame(columns=["PULocationID", "PU_Zone"])

display(pu_zone_lookup.head())

del pu_zone_lookups
gc.collect()

## 9. Полная сетка: все часы × все районы

Если в каком-то районе в какой-то час не было поездок, после merge будет пропуск.

Его заполним:

```text
demand = 0
revenue = 0
```

In [ ]:
all_hours = pd.date_range(
    start=hourly_real["date_hour"].min(),
    end=hourly_real["date_hour"].max(),
    freq="h"
)

hours_df = pd.DataFrame({"date_hour": all_hours})

all_locations = (
    hourly_real[["PULocationID"]]
    .drop_duplicates()
    .sort_values("PULocationID")
    .reset_index(drop=True)
)

all_locations["PULocationID"] = all_locations["PULocationID"].astype("int16")

full_grid = hours_df.merge(all_locations, how="cross")
full_grid["PULocationID"] = full_grid["PULocationID"].astype("int16")

print("Количество часов:", len(all_hours))
print("Количество районов:", len(all_locations))
print("Размер полной сетки:", full_grid.shape)

display(full_grid.head())

## 10. Добавляем demand/revenue и заполняем нулями

In [ ]:
hourly = full_grid.merge(
    hourly_real,
    on=["date_hour", "PULocationID"],
    how="left"
)

hourly["demand"] = hourly["demand"].fillna(0).astype("int32")
hourly["revenue"] = hourly["revenue"].fillna(0).astype("float32")

# Погода зависит от часа, а не от района.
weather_hourly = (
    hourly_real
    .groupby("date_hour")[weather_features]
    .mean()
    .reset_index()
)

for col in weather_features:
    weather_hourly[col] = weather_hourly[col].astype("float32")

hourly = hourly.drop(columns=weather_features, errors="ignore")

hourly = hourly.merge(
    weather_hourly,
    on="date_hour",
    how="left"
)

hourly[weather_features] = hourly[weather_features].ffill().bfill()

for col in weather_features:
    hourly[col] = hourly[col].astype("float32")

hourly = hourly.merge(
    pu_zone_lookup,
    on="PULocationID",
    how="left"
)

hourly = hourly.sort_values(["PULocationID", "date_hour"]).reset_index(drop=True)

del full_grid
del hourly_real
del weather_hourly
gc.collect()

print("Размер hourly:", hourly.shape)
print("Минимальная revenue:", hourly["revenue"].min())
print("Максимальная revenue:", hourly["revenue"].max())

display(hourly.head())

## 11. Сохраняем подготовленную таблицу

In [ ]:
prepared_path = TABLES_DIR / "catboost_hourly_revenue_all_hours_all_zones.parquet"

hourly.to_parquet(prepared_path, index=False)

print("Подготовленная таблица сохранена:")
print(prepared_path)

## 12. Признаки времени

In [ ]:
hourly["month"] = hourly["date_hour"].dt.month.astype("int8")
hourly["day"] = hourly["date_hour"].dt.day.astype("int8")
hourly["hour"] = hourly["date_hour"].dt.hour.astype("int8")
hourly["dayofweek"] = hourly["date_hour"].dt.dayofweek.astype("int8")
hourly["is_weekend"] = (hourly["dayofweek"] >= 5).astype("int8")

hourly["hour_sin"] = np.sin(2 * np.pi * hourly["hour"] / 24).astype("float32")
hourly["hour_cos"] = np.cos(2 * np.pi * hourly["hour"] / 24).astype("float32")

hourly["dayofweek_sin"] = np.sin(2 * np.pi * hourly["dayofweek"] / 7).astype("float32")
hourly["dayofweek_cos"] = np.cos(2 * np.pi * hourly["dayofweek"] / 7).astype("float32")

display(hourly.head())

## 13. Лаги

Для каждого района считаем прошлый спрос и прошлую выручку.

Так как у нас полная сетка часов, `shift(1)` действительно означает предыдущий час.

In [ ]:
hourly = hourly.sort_values(["PULocationID", "date_hour"]).reset_index(drop=True)

group = hourly.groupby("PULocationID", sort=False)

hourly["lag_1_revenue"] = group["revenue"].shift(1)
hourly["lag_2_revenue"] = group["revenue"].shift(2)
hourly["lag_3_revenue"] = group["revenue"].shift(3)
hourly["lag_24_revenue"] = group["revenue"].shift(24)
hourly["lag_168_revenue"] = group["revenue"].shift(168)

hourly["lag_1_demand"] = group["demand"].shift(1)
hourly["lag_24_demand"] = group["demand"].shift(24)
hourly["lag_168_demand"] = group["demand"].shift(168)

hourly["rolling_mean_3_revenue"] = group["revenue"].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)

hourly["rolling_mean_6_revenue"] = group["revenue"].transform(
    lambda x: x.shift(1).rolling(window=6, min_periods=1).mean()
)

hourly["rolling_mean_24_revenue"] = group["revenue"].transform(
    lambda x: x.shift(1).rolling(window=24, min_periods=1).mean()
)

lag_columns = [
    "lag_1_revenue",
    "lag_2_revenue",
    "lag_3_revenue",
    "lag_24_revenue",
    "lag_168_revenue",
    "lag_1_demand",
    "lag_24_demand",
    "lag_168_demand",
    "rolling_mean_3_revenue",
    "rolling_mean_6_revenue",
    "rolling_mean_24_revenue",
]

hourly[lag_columns] = hourly[lag_columns].fillna(0)

for col in lag_columns:
    hourly[col] = hourly[col].astype("float32")

display(hourly.head())

## 14. PULocationID как категориальный признак

Для CatBoost не нужно делать one-hot encoding.

Мы передадим `PULocationID` как категориальный признак.

In [ ]:
# CatBoost ожидает категориальные признаки как строку или category.
hourly["PULocationID_cat"] = hourly["PULocationID"].astype(str)

display(hourly[["PULocationID", "PULocationID_cat", "PU_Zone"]].head())

## 15. Train/test по времени

In [ ]:
unique_hours = np.array(sorted(hourly["date_hour"].unique()))

split_index = int(len(unique_hours) * 0.8)
split_time = unique_hours[split_index]

train = hourly[hourly["date_hour"] < split_time].copy()
test = hourly[hourly["date_hour"] >= split_time].copy()

print("Граница train/test:", split_time)
print("Train:", train.shape)
print("Test:", test.shape)

print("Train даты:", train["date_hour"].min(), "—", train["date_hour"].max())
print("Test даты:", test["date_hour"].min(), "—", test["date_hour"].max())

## 16. Признаки модели

In [ ]:
feature_columns = [
    "PULocationID_cat",

    "month",
    "day",
    "hour",
    "dayofweek",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dayofweek_sin",
    "dayofweek_cos",

    "demand",

    "lag_1_revenue",
    "lag_2_revenue",
    "lag_3_revenue",
    "lag_24_revenue",
    "lag_168_revenue",

    "lag_1_demand",
    "lag_24_demand",
    "lag_168_demand",

    "rolling_mean_3_revenue",
    "rolling_mean_6_revenue",
    "rolling_mean_24_revenue",
] + weather_features

feature_columns = [col for col in feature_columns if col in hourly.columns]

cat_features = ["PULocationID_cat"]

print("Количество признаков:", len(feature_columns))
print("Признаки:")
print(feature_columns)

print("Категориальные признаки:")
print(cat_features)

## 17. Готовим Pool для CatBoost

In [ ]:
X_train = train[feature_columns].copy()
y_train = train["revenue"].astype("float32").copy()

X_test = test[feature_columns].copy()
y_test = test["revenue"].astype("float32").copy()

train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=cat_features
)

test_pool = Pool(
    data=X_test,
    label=y_test,
    cat_features=cat_features
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

## 18. Обучаем CatBoostRegressor

Это градиентный бустинг.

Если обучение идёт долго, можно уменьшить:

```python
iterations=300
depth=6
```

In [ ]:
catboost_model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",

    iterations=600,
    learning_rate=0.05,
    depth=8,

    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,

    od_type="Iter",
    od_wait=50,

    verbose=100,
    allow_writing_files=False,
)

print("Обучаем CatBoost...")
catboost_model.fit(
    train_pool,
    eval_set=test_pool,
    use_best_model=True
)

print("Готово")

## 19. Оцениваем регрессию по выручке

In [ ]:
pred_revenue = catboost_model.predict(test_pool)

pred_revenue = np.asarray(pred_revenue, dtype="float64")
pred_revenue = np.nan_to_num(pred_revenue, nan=0.0, posinf=300000, neginf=0.0)
pred_revenue = np.clip(pred_revenue, 0, 300000)

mae = mean_absolute_error(y_test, pred_revenue)
rmse = mean_squared_error(y_test, pred_revenue) ** 0.5
r2 = r2_score(y_test, pred_revenue)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

metrics_regression = pd.DataFrame([
    {
        "model": "CatBoostRegressor",
        "target": "revenue",
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    }
])

display(metrics_regression)

metrics_regression_path = TABLES_DIR / "catboost_profitable_zone_revenue_regression_metrics.csv"
metrics_regression.to_csv(metrics_regression_path, index=False)

print("Метрики регрессии сохранены:")
print(metrics_regression_path)

## 20. Выбираем самый прибыльный район по каждому часу

Для каждого часа выбираем:

```text
реальный лучший район = max(revenue)
предсказанный лучший район = max(predicted_revenue)
```

In [ ]:
test_predictions = test[[
    "date_hour",
    "PULocationID",
    "PU_Zone",
    "revenue",
    "demand",
]].copy()

test_predictions["predicted_revenue"] = pred_revenue

actual_best = (
    test_predictions
    .sort_values(["date_hour", "revenue"], ascending=[True, False])
    .groupby("date_hour", as_index=False)
    .first()
    .rename(columns={
        "PULocationID": "actual_best_PULocationID",
        "PU_Zone": "actual_best_PU_Zone",
        "revenue": "actual_best_revenue",
        "demand": "actual_best_demand",
    })
)

predicted_best = (
    test_predictions
    .sort_values(["date_hour", "predicted_revenue"], ascending=[True, False])
    .groupby("date_hour", as_index=False)
    .first()
    .rename(columns={
        "PULocationID": "predicted_best_PULocationID",
        "PU_Zone": "predicted_best_PU_Zone",
        "revenue": "predicted_best_actual_revenue",
        "demand": "predicted_best_actual_demand",
        "predicted_revenue": "predicted_best_revenue",
    })
)

best_by_hour = actual_best.merge(
    predicted_best,
    on="date_hour",
    how="inner"
)

best_by_hour["is_correct_zone"] = (
    best_by_hour["actual_best_PULocationID"]
    == best_by_hour["predicted_best_PULocationID"]
)

top1_accuracy = best_by_hour["is_correct_zone"].mean()

print("Top-1 accuracy по самому прибыльному району:", top1_accuracy)

display(best_by_hour.head())

best_by_hour_path = TABLES_DIR / "catboost_best_profitable_zone_by_hour_test.csv"
best_by_hour.to_csv(best_by_hour_path, index=False)

test_predictions_path = TABLES_DIR / "catboost_revenue_predictions_test.csv"
test_predictions.to_csv(test_predictions_path, index=False)

print("Таблица лучших районов по часам сохранена:")
print(best_by_hour_path)

print("Все предсказания по тесту сохранены:")
print(test_predictions_path)

## 21. Сохраняем модель

In [ ]:
model_path = MODELS_DIR / "catboost_most_profitable_zone_revenue_model.cbm"
features_path = MODELS_DIR / "catboost_most_profitable_zone_features.joblib"

catboost_model.save_model(model_path)

feature_info = {
    "feature_columns": feature_columns,
    "cat_features": cat_features,
    "weather_features": weather_features,
}

joblib.dump(feature_info, features_path)

print("CatBoost-модель сохранена:")
print(model_path)

print("Признаки сохранены:")
print(features_path)

## 22. Прогноз самого прибыльного района на следующий час

Берём последний известный час для каждого района и прогнозируем следующий час.

In [ ]:
last_hour = hourly["date_hour"].max()
next_hour = last_hour + pd.Timedelta(hours=1)

future = (
    hourly
    .sort_values(["PULocationID", "date_hour"])
    .groupby("PULocationID", sort=False)
    .tail(1)
    .copy()
)

future["date_hour"] = next_hour

future["month"] = future["date_hour"].dt.month.astype("int8")
future["day"] = future["date_hour"].dt.day.astype("int8")
future["hour"] = future["date_hour"].dt.hour.astype("int8")
future["dayofweek"] = future["date_hour"].dt.dayofweek.astype("int8")
future["is_weekend"] = (future["dayofweek"] >= 5).astype("int8")

future["hour_sin"] = np.sin(2 * np.pi * future["hour"] / 24).astype("float32")
future["hour_cos"] = np.cos(2 * np.pi * future["hour"] / 24).astype("float32")

future["dayofweek_sin"] = np.sin(2 * np.pi * future["dayofweek"] / 7).astype("float32")
future["dayofweek_cos"] = np.cos(2 * np.pi * future["dayofweek"] / 7).astype("float32")

# Последние известные значения становятся лагами для следующего часа.
future["lag_1_revenue"] = future["revenue"].astype("float32")
future["lag_1_demand"] = future["demand"].astype("float32")

future_pool = Pool(
    data=future[feature_columns],
    cat_features=cat_features
)

future["predicted_revenue"] = catboost_model.predict(future_pool)

future["predicted_revenue"] = np.nan_to_num(
    future["predicted_revenue"],
    nan=0.0,
    posinf=300000,
    neginf=0.0
)

future["predicted_revenue"] = np.clip(
    future["predicted_revenue"],
    0,
    300000
)

future_predictions = future[[
    "date_hour",
    "PULocationID",
    "PU_Zone",
    "predicted_revenue",
]].copy()

future_predictions = future_predictions.sort_values(
    "predicted_revenue",
    ascending=False
)

display(future_predictions.head(20))

best_next_hour = future_predictions.head(1).copy()

print("Самый прибыльный район на следующий час по прогнозу:")
display(best_next_hour)

future_predictions_path = TABLES_DIR / "catboost_next_hour_profitable_zone_predictions.csv"
best_next_hour_path = TABLES_DIR / "catboost_best_next_hour_profitable_zone.csv"

future_predictions.to_csv(future_predictions_path, index=False)
best_next_hour.to_csv(best_next_hour_path, index=False)

print("Все прогнозы районов сохранены:")
print(future_predictions_path)

print("Лучший район сохранён:")
print(best_next_hour_path)

## Итог

Этот ноутбук обучает CatBoost-модель:

```text
для каждого часа прогнозирует выручку по каждому району,
а затем выбирает район с максимальной predicted_revenue
```

Главные выходные файлы:

```text
models/catboost_most_profitable_zone_revenue_model.cbm

reports/tables/catboost_best_profitable_zone_by_hour_test.csv
reports/tables/catboost_best_next_hour_profitable_zone.csv
reports/tables/catboost_next_hour_profitable_zone_predictions.csv
reports/tables/catboost_profitable_zone_revenue_regression_metrics.csv
```